In [ ]:
!pip install rouge_score
!pip install openai
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import numpy as np
import re

### Load Data

In this version, we filtered the data to eliminate things the model struggled with:
1. Code heavy answers
2. Long complex prompts or answers
3. Very short promprts or answers

In [3]:
df = pd.read_csv("prompt_answer_pairs_clean.csv")
print(f"Original: {len(df)} rows")

# 1. Filter rows where answer is mostly code blocks to remove blocks
def clean_answer(text):
    text = re.sub(r'\[CODE_BLOCK_\d+\]', '', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["answer"] = df["answer"].apply(clean_answer)
print(f"After code block filter: {len(df)} rows")

# 2. Drop rows where answer became too short after removing code blocks
df = df[df["answer"].str.split().str.len() >= 10]
print(f"After dropping short answers: {len(df)} rows")

# 3. Filter out very long prompts (over 50 words)
df = df[df["prompt"].str.split().str.len() <= 50]
print(f"After long prompt filter: {len(df)} rows")

# 4. Filter out very short prompts (less than 3 words)
df = df[df["prompt"].str.split().str.len() >= 3]
print(f"After short prompt filter: {len(df)} rows")

# 5. Filter out very long answers (over 400 words)
df = df[df["answer"].str.split().str.len() <= 400]
print(f"After long answer filter: {len(df)} rows")

df.to_csv("prompt_answer_pairs_filtered.csv", index=False, encoding="utf-8")

Original: 8058 rows
After code block filter: 8058 rows
After dropping short answers: 7509 rows
After long prompt filter: 4650 rows
After short prompt filter: 4621 rows
After long answer filter: 4315 rows


In [5]:
df = pd.read_csv("prompt_answer_pairs_context.csv")[["context_input", "prompt", "answer"]].dropna()

# Convert to strings
df["prompt"] = df["prompt"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df["context_input"] = df["context_input"].astype(str).str.strip()

# 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

### Tokenize data

In [16]:
def tokenize_data(df):
  # Input is the answer, tokenizes answer text
  inputs = tokenizer(
      df["context_input"].tolist(),
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
    )
  # Target is the prompt, tokenizes prompt text
  targets = tokenizer(
      df["prompt"].tolist(),
      max_length=128,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  labels = targets["input_ids"]
  # Replace padding tokens with -100 so model does not try to predict them
  labels[labels == tokenizer.pad_token_id] = -100

  # Create tokenized dataset
  dataset = torch.utils.data.TensorDataset(
      inputs["input_ids"],
      inputs["attention_mask"],
      labels
  )
  return dataset

# Baseline
Gets the baseline score

In [17]:
"""
Predict the prompt from a GPT output, then score it with ROUGE-L.

Usage:
    python predict_prompt.py --inp "The capital of France is Paris." --out "What is the capital of France?"
"""

import argparse
from openai import OpenAI

# pip install openai rouge-score
from rouge_score import rouge_scorer


def predict_prompt(client: OpenAI, inp: str) -> str:
    """Ask GPT to predict the prompt that produced `inp`."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": f"Given a GPT model outputted: {inp}, predict the prompt.",
            }
        ],
    )
    return response.choices[0].message.content.strip()


def rouge_l(hypothesis: str, reference: str) -> float:
    """Return the ROUGE-L F1 score between hypothesis and reference."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    return scores["rougeL"].fmeasure

def numberItemsGEQThan(i,L):
  return sum(1 for n in L if n >= i);

### Main method for getting baseline

In [ ]:
skip = 100 #this will skip to every 100th row
startingPoint = 0 #resume where you left off, if need be
df = pd.read_csv("prompt_answer_pairs_filtered.csv")
ROGUES = []
client = OpenAI(api_key="YOURAPIKEY")
for idx, row in df.iterrows():
  if (idx < startingPoint):
    continue
  if (idx%skip == 0):
    GPTRes = predict_prompt(client, row["answer"])
    ROGUES.append(rouge_l(row["prompt"],GPTRes))


print("Average ROGUE: ", sum(ROGUES)/len(ROGUES))




print("Cumultive Distribution: ")
print(f">0.00: {numberItemsGEQThan(0,ROGUES)}")
print(f">0.05: {numberItemsGEQThan(0.05,ROGUES)}")
print(f">0.10: {numberItemsGEQThan(0.1,ROGUES)}")
print(f">0.15: {numberItemsGEQThan(0.15,ROGUES)}")
print(f">0.20: {numberItemsGEQThan(0.2,ROGUES)}")
print(f">0.25: {numberItemsGEQThan(0.25,ROGUES)}")
print(f">0.30: {numberItemsGEQThan(0.3,ROGUES)}")
print(f">0.35: {numberItemsGEQThan(0.35,ROGUES)}")
print(f">0.40: {numberItemsGEQThan(0.4,ROGUES)}")
print(f">0.45: {numberItemsGEQThan(0.45,ROGUES)}")
print(f">0.50: {numberItemsGEQThan(0.5,ROGUES)}")
print(f">0.55: {numberItemsGEQThan(0.55,ROGUES)}")
print(f">0.60: {numberItemsGEQThan(0.6,ROGUES)}")
print(f">0.65: {numberItemsGEQThan(0.65,ROGUES)}")
print(f">0.70: {numberItemsGEQThan(0.7,ROGUES)}")
print(f">0.75: {numberItemsGEQThan(0.75,ROGUES)}")
print(f">0.80: {numberItemsGEQThan(0.8,ROGUES)}")
print(f">0.85: {numberItemsGEQThan(0.85,ROGUES)}")
print(f">0.90: {numberItemsGEQThan(0.9,ROGUES)}")
print(f">0.95: {numberItemsGEQThan(0.95,ROGUES)}")




RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}